# 대출 승인 예측 ML 모델

고객의 대출 승인 여부를 예측하는 이진 분류 모델을 구축합니다.  
    • 데이터: training_data.csv (5,000건, 13개 변수)  
    • 모델: Logistic Regression, Random Forest, XGBoost, LightGBM 중에 2개 선택해서 비교  
    • 평가 지표: 정확도, 정밀도, 재현율, F1 Score (불균형 데이터 대응)  

## 선택한 모델
- Logistic Regression
  - 대출 승인은 금융 도메인 — 모델의 해석 가능성이 중요 (규제/감사 대응)
  - 베이스라인 역할: 복잡한 모델과 성능 비교 기준점
  - 수치형 피처만 있어 전처리 부담 없음
- LightGBM
  - 5,000행 정도의 테이블형 데이터에서 Gradient Boosting 계열이 압도적으로 강함
  - XGBoost보다 학습 속도 빠름, 메모리 효율 좋음
  - class_weight / scale_pos_weight로 클래스 불균형(71:29) 쉽게 대응
  - Random Forest보다 일반적으로 성능 우위

# 0. Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 데이터 분석

### 1-1. 데이터 탐색 및 결측치 확인

In [ ]:
df = pd.read_csv('training_data.csv')
print(df.shape)
print(df.info())
print('------')
# 클래스 불균형 확인
print(df['승인여부'].value_counts(normalize=True))
# 결측치 확인
print(df.isnull().sum())


(5000, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   나이      5000 non-null   int64
 1   연봉      5000 non-null   int64
 2   재산      5000 non-null   int64
 3   예금      5000 non-null   int64
 4   자동차배기량  5000 non-null   int64
 5   자녀수     5000 non-null   int64
 6   신청금액    5000 non-null   int64
 7   신청기간월   5000 non-null   int64
 8   학력점수    5000 non-null   int64
 9   결혼점수    5000 non-null   int64
 10  회사규모점수  5000 non-null   int64
 11  대출분류점수  5000 non-null   int64
 12  승인여부    5000 non-null   int64
dtypes: int64(13)
memory usage: 507.9 KB
None
------
승인여부
1    0.7118
0    0.2882
Name: proportion, dtype: float64
나이        0
연봉        0
재산        0
예금        0
자동차배기량    0
자녀수       0
신청금액      0
신청기간월     0
학력점수      0
결혼점수      0
회사규모점수    0
대출분류점수    0
승인여부      0
dtype: int64


### 1-2. 전처리

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('승인여부', axis=1)
y = df['승인여부']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # stratify로 비율 유지
)

# Logistic Regression은 스케일링 필요
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


1-3. 모델 학습

In [ ]:
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)

lgbm = LGBMClassifier(class_weight='balanced', random_state=42, n_estimators=300)
lgbm.fit(X_train, y_train)  # LightGBM은 스케일링 불필요
